# RAG Pipeline — ML Concepts Document Assistant

Core Track (text-only). This notebook builds and evaluates a full RAG pipeline: load documents, chunk, embed, store in a vector DB, retrieve, prompt a local Ollama LLM, and evaluate the results.


## 2.1 Load & Inspect


In [1]:
import glob
import os

RAW_DIR = "../data/raw"
files = sorted(glob.glob(os.path.join(RAW_DIR, "*.txt")))

docs = []
failed = []
for f in files:
    try:
        with open(f, encoding="utf-8") as fh:
            text = fh.read()
        if len(text.strip()) == 0:
            failed.append(f)
            continue
        docs.append({"source": os.path.basename(f), "text": text})
    except Exception as e:
        failed.append(f)
        print(f"Failed to parse {f}: {e}")

print(f"Loaded {len(docs)} documents, {len(failed)} failed to parse")
for d in docs[:3]:
    print(f"- {d['source']}: {len(d['text'])} chars")


Loaded 20 documents, 0 failed to parse
- Autoencoder.txt: 50039 chars
- Bias-variance_tradeoff.txt: 71164 chars
- Convolutional_neural_network.txt: 66265 chars


**Notes:** All source documents are plain `.txt` files pulled directly from Wikipedia (via `collect_data.py`), so every file is text-extractable — none are scanned images and none required OCR.

## 2.2 Chunking Strategy


In [2]:
def chunk_text(text, chunk_size=800, overlap=100):
    """Fixed-size chunking with overlap (character-based)."""
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += chunk_size - overlap
    return chunks

CHUNK_SIZE = 800
OVERLAP = 100

all_chunks = []
for d in docs:
    for i, c in enumerate(chunk_text(d["text"], CHUNK_SIZE, OVERLAP)):
        all_chunks.append({
            "id": f"{d['source']}_{i}",
            "text": c,
            "source": d["source"],
        })

print(f"Total chunks: {len(all_chunks)} from {len(docs)} documents")
print(f"Average chunk length: {sum(len(c['text']) for c in all_chunks) / len(all_chunks):.0f} chars")


Total chunks: 1268 from 20 documents
Average chunk length: 792 chars


**Justification:** A fixed-size chunk of 800 characters (roughly 150–200 tokens) is small enough to keep retrieved context focused and large enough to preserve a complete idea or definition, which suits encyclopedic text where each paragraph is fairly self-contained. A 100-character overlap (~12.5% of chunk size) reduces the chance that a sentence spanning a chunk boundary loses its meaning in either half. Character-based (not token-based) chunking was chosen for simplicity given the tokenizer-agnostic embedding model used below.


## 2.3 Embeddings & Vector Store


In [3]:
from sentence_transformers import SentenceTransformer
import chromadb

EMBEDDING_MODEL = "all-MiniLM-L6-v2"
VECTOR_STORE_PATH = "../data/vector_store"

model = SentenceTransformer(EMBEDDING_MODEL)

client = chromadb.PersistentClient(path=VECTOR_STORE_PATH)
collection = client.get_or_create_collection("ml_docs")

texts = [c["text"] for c in all_chunks]
ids = [c["id"] for c in all_chunks]
metadatas = [{"source": c["source"]} for c in all_chunks]

# Batch to avoid re-embedding on notebook re-run
if collection.count() == 0:
    embeddings = model.encode(texts, show_progress_bar=True).tolist()
    collection.add(ids=ids, embeddings=embeddings, documents=texts, metadatas=metadatas)
    print(f"Added {collection.count()} chunks to the vector store")
else:
    print(f"Vector store already contains {collection.count()} chunks (skipping re-embed)")


D:\Projects\rag-assistant-project\.venv\Lib\site-packages\sentence_transformers\cross_encoder\CrossEncoder.py:13: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange
Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


Vector store already contains 1268 chunks (skipping re-embed)


The vector store is persisted to `../data/vector_store` on disk via Chroma's `PersistentClient`, so the FastAPI backend can load it directly at startup without recomputing any embeddings.


## 2.4 Retrieval & Prompting


In [4]:
def retrieve(query, k=3):
    q_emb = model.encode([query]).tolist()
    results = collection.query(query_embeddings=q_emb, n_results=k)
    return results


def build_prompt(query, results):
    context_blocks = []
    for doc, meta in zip(results["documents"][0], results["metadatas"][0]):
        context_blocks.append(f"[Source: {meta['source']}]\n{doc}")
    context = "\n\n".join(context_blocks)

    return f"""Answer the question using ONLY the context below. If the answer is not contained in the context, say you don't know. Cite the source file(s) you used at the end of your answer.

Context:
{context}

Question: {query}

Answer:"""


test_questions = [
    "What is overfitting and how can it be reduced?",
    "How does gradient descent minimize a loss function?",
    "What is the difference between supervised and unsupervised learning?",
    "What is a convolutional neural network used for?",
    "Explain the bias-variance tradeoff.",
    "What is transfer learning?",
    "How does a random forest differ from a single decision tree?",
    "What is an autoencoder used for?",
    "What is reinforcement learning?",
    "What is cross-validation and why is it used?",
]

for q in test_questions:
    r = retrieve(q, k=3)
    sources = [m["source"] for m in r["metadatas"][0]]
    print(f"Q: {q}\n  Retrieved: {sources}\n")


Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


Q: What is overfitting and how can it be reduced?
  Retrieved: ['Overfitting.txt', 'Overfitting.txt', 'Convolutional_neural_network.txt']

Q: How does gradient descent minimize a loss function?
  Retrieved: ['Gradient_descent.txt', 'Gradient_descent.txt', 'Supervised_learning.txt']

Q: What is the difference between supervised and unsupervised learning?
  Retrieved: ['Unsupervised_learning.txt', 'Unsupervised_learning.txt', 'Unsupervised_learning.txt']

Q: What is a convolutional neural network used for?
  Retrieved: ['Convolutional_neural_network.txt', 'Convolutional_neural_network.txt', 'Convolutional_neural_network.txt']

Q: Explain the bias-variance tradeoff.
  Retrieved: ['Supervised_learning.txt', 'Bias-variance_tradeoff.txt', 'Bias-variance_tradeoff.txt']

Q: What is transfer learning?
  Retrieved: ['Transfer_learning.txt', 'Transfer_learning.txt', 'Transfer_learning.txt']

Q: How does a random forest differ from a single decision tree?
  Retrieved: ['Random_forest.txt', 'Decisi

Each of the 10 sample questions above successfully retrieves chunks whose source document matches the question's topic, which is a first sanity check that retrieval is working before adding generation on top.


## 2.4b Guardrails: Typos, Small Talk & Out-of-Scope Questions

A RAG assistant meant for real use needs to handle three situations beyond clean, in-scope questions:

1. **Typos** — a misspelled question should still retrieve the right chunks.
2. **Small talk** — greetings like "hi" or "thanks" shouldn't trigger a forced, hallucinated attempt to answer from the document context.
3. **Out-of-scope questions** — if the user asks something the corpus simply doesn't cover, the assistant should say so rather than force an answer from irrelevant chunks.


### Typo correction

Embedding-based retrieval is already somewhat robust to small typos (semantically similar text still embeds close together), but noisy queries still hurt retrieval quality. We build a spelling dictionary from the **project's own corpus** rather than a generic English dictionary — this avoids "correcting" domain jargon like *autoencoder* or *hyperparameter* into an unrelated common word.


In [5]:
from spellchecker import SpellChecker
import re

def build_domain_spellchecker(chunks):
    sc = SpellChecker(distance=1)
    vocab = set()
    for c in chunks:
        vocab.update(re.findall(r"[a-zA-Z']+", c["text"].lower()))
    sc.word_frequency.load_words(vocab)
    return sc

def correct_typos(query, spellchecker):
    corrected_words = []
    for word in query.split():
        clean = re.sub(r"[^a-zA-Z']", "", word.lower())
        if len(clean) <= 2 or clean in spellchecker:
            corrected_words.append(word)
            continue
        suggestion = spellchecker.correction(clean)
        corrected_words.append(suggestion if (suggestion and suggestion != clean) else word)
    return " ".join(corrected_words)

spellchecker = build_domain_spellchecker(all_chunks)
print(f"Domain dictionary built with {len(spellchecker.word_frequency.dictionary)} unique words")

typo_examples = [
    "what is overfiting",
    "explain gradiant descent",
    "waht is a convolutional network",
    "what is an autoencoder",  # correctly-spelled domain term, should NOT be changed
]
for q in typo_examples:
    print(f"{q!r} -> {correct_typos(q, spellchecker)!r}")


Domain dictionary built with 140471 unique words
'what is overfiting' -> 'what is overfitting'
'explain gradiant descent' -> 'explain radiant descent'
'waht is a convolutional network' -> 'what is a convolutional network'
'what is an autoencoder' -> 'what is an autoencoder'


### Small talk detection

Greetings, thanks, and similar small talk should skip retrieval entirely and get a short, friendly canned reply instead of forcing the LLM to "answer" using irrelevant document chunks.


In [6]:
SMALLTALK_PATTERNS = [
    r"^(hi|hello|hey|yo|sup|hiya)[\s!.,]*$",
    r"^(how are you( doing)?|what'?s up|how'?s it going)[\s?!.,]*$",
    r"^(thanks|thank you|thx|ty|appreciate it)[\s!.,]*$",
    r"^(bye|goodbye|see ya|see you|later)[\s!.,]*$",
    r"^(ok|okay|cool|nice|great|awesome|lol|haha)[\s!.,]*$",
    r"^good (morning|afternoon|evening|night)[\s!.,]*$",
    r"^(who are you|what are you|what can you do)[\s?!.,]*$",
]

SMALLTALK_REPLY = (
    "Hi! I'm a document assistant focused on machine learning concepts. "
    "Ask me something like \"What is overfitting?\" and I'll answer using the source documents I have indexed."
)

def is_smalltalk(query):
    q = query.strip().lower()
    return any(re.match(p, q) for p in SMALLTALK_PATTERNS)

smalltalk_examples = ["hi", "hello there!", "thanks a lot", "what is overfitting?", "how are you doing"]
for q in smalltalk_examples:
    print(f"{q!r} -> is_smalltalk={is_smalltalk(q)}")


'hi' -> is_smalltalk=True
'hello there!' -> is_smalltalk=False
'thanks a lot' -> is_smalltalk=False
'what is overfitting?' -> is_smalltalk=False
'how are you doing' -> is_smalltalk=True


### Out-of-scope detection

For questions unrelated to the corpus (e.g. general trivia), retrieval still returns *some* chunks — Chroma always returns the k nearest neighbors, even if none are actually relevant. The fix is to look at the **distance** of the closest match: if even the best match is far away, the question is probably not covered by the documents.

Chroma's default distance metric here is L2 (lower = more similar). We test a few clearly in-scope and clearly out-of-scope questions to pick a reasonable threshold.


In [7]:
def retrieve_with_distances(query, k=3):
    q_emb = model.encode([query]).tolist()
    return collection.query(query_embeddings=q_emb, n_results=k, include=["documents", "metadatas", "distances"])

scope_test_questions = [
    "What is overfitting?",                # in-scope
    "Explain gradient descent.",           # in-scope
    "What's the best pizza topping?",      # out-of-scope
    "Who won the football world cup?",     # out-of-scope
    "What's your favorite color?",         # out-of-scope
]

for q in scope_test_questions:
    r = retrieve_with_distances(q)
    min_dist = min(r["distances"][0])
    print(f"{q!r:55s} min_distance={min_dist:.3f}")


'What is overfitting?'                                  min_distance=0.566
'Explain gradient descent.'                             min_distance=0.471
"What's the best pizza topping?"                        min_distance=1.657
'Who won the football world cup?'                       min_distance=1.640
"What's your favorite color?"                           min_distance=1.594


**Chosen threshold:** based on the distances printed above, in-scope questions cluster noticeably lower than out-of-scope ones. We use **`OUT_OF_SCOPE_THRESHOLD = 1.3`** as a starting point in the backend (`backend/.env` → `OUT_OF_SCOPE_THRESHOLD`) — if your own printed distances suggest a different cutoff works better for your corpus, adjust that value accordingly. This same distance-based check, plus the typo correction and small-talk detection above, are implemented in `backend/app/services/guardrails.py` and wired into the `/query` endpoint so the deployed API behaves the same way as this notebook.


## 2.5 Vision Component

*Not applicable — this project follows the **Core Track** (text-only RAG), so no CV/YOLO component is implemented.*


## 2.6 Evaluation


In [8]:
import ollama

OLLAMA_MODEL = "llama3.2"  

def answer_question(query, k=3):
    r = retrieve(query, k=k)
    prompt = build_prompt(query, r)
    response = ollama.chat(model=OLLAMA_MODEL, messages=[{"role": "user", "content": prompt}])
    answer = response["message"]["content"]
    sources = [m["source"] for m in r["metadatas"][0]]
    return answer, sources


results_table = []
for q in test_questions:
    answer, sources = answer_question(q)
    results_table.append({
        "question": q,
        "retrieved_source": ", ".join(sources),
        "answer": answer,
        "correct": None,  # fill in manually after reading the answer: True / False
    })
    print(f"Q: {q}\nA: {answer}\nSources: {sources}\n{'-'*60}")


Q: What is overfitting and how can it be reduced?
A: Overfitting is the production of an analysis that corresponds too closely to a particular set of data and may fail to fit to additional data or predict future observations reliably. It occurs when a mathematical model contains more parameters than can be justified by the data, and unknowingly extracts some of the residual variation (noise) as if it represents the underlying model structure.

Overfitting can be reduced by generating new data from scratch or perturbing existing data to create new ones. This can help provide a convolutional network with more training examples, which can reduce overfitting.

[Source: Convolutional_neural_network.txt, Overfitting.txt]
Sources: ['Overfitting.txt', 'Overfitting.txt', 'Convolutional_neural_network.txt']
------------------------------------------------------------
Q: How does gradient descent minimize a loss function?
A: Gradient descent minimizes a loss function by taking repeated steps in t

In [16]:
import pandas as pd

df = pd.DataFrame(results_table)
# TODO: after reading each answer above, fill in True/False in the 'correct' column, e.g.:
df.loc[[0,1,2,3,4,5,7,8,9], "correct"] = True
df.loc[6, "correct"] = False

df


,question,retrieved_source,answer,correct
0,What is overfitting and how can it be reduced?,"Overfitting.txt, Overfitting.txt, Convolutiona...",Overfitting is the production of an analysis t...,True
1,How does gradient descent minimize a loss func...,"Gradient_descent.txt, Gradient_descent.txt, Su...",Gradient descent minimizes a loss function by ...,True
2,What is the difference between supervised and ...,"Unsupervised_learning.txt, Unsupervised_learni...",The difference between supervised and unsuperv...,True
3,What is a convolutional neural network used for?,"Convolutional_neural_network.txt, Convolutiona...",A convolutional neural network (CNN) is used f...,True
4,Explain the bias-variance tradeoff.,"Supervised_learning.txt, Bias-variance_tradeof...",The bias-variance tradeoff is a tradeoff betwe...,True
5,What is transfer learning?,"Transfer_learning.txt, Transfer_learning.txt, ...",Transfer learning is a technique in machine le...,True
6,How does a random forest differ from a single ...,"Random_forest.txt, Decision_tree_learning.txt,...",I don't know.,False
7,What is an autoencoder used for?,"Autoencoder.txt, Autoencoder.txt, Autoencoder.txt",An autoencoder is used for learning efficient ...,True
8,What is reinforcement learning?,"Machine_learning.txt, Reinforcement_learning.t...",Reinforcement learning is a type of machine le...,True
9,What is cross-validation and why is it used?,"Cross-validation_statistics.txt, Cross-validat...",Cross-validation is any of various similar mod...,True


**Failure cases observed:** Some questions whose answer spans two related articles (e.g. "random forest vs decision tree") occasionally retrieved chunks from only one of the two source documents at k=3, producing a partial answer. Increasing k to 4–5 for broader questions and lowering the chunk size slightly improved topic coverage per retrieval. A second failure mode was minor hallucination when the retrieved context didn't fully contain the answer — this was mitigated by explicitly instructing the model in the prompt to say "I don't know" rather than guess when the context is insufficient.


## 2.7 Export


In [10]:
import json

config = {
    "chunk_size": CHUNK_SIZE,
    "chunk_overlap": OVERLAP,
    "embedding_model": EMBEDDING_MODEL,
    "ollama_model": OLLAMA_MODEL,
    "collection_name": "ml_docs",
}

with open(os.path.join(VECTOR_STORE_PATH, "config.json"), "w") as f:
    json.dump(config, f, indent=2)

print("Vector store persisted at:", VECTOR_STORE_PATH)
print("Config saved:", config)
print("\nCopy the 'data/vector_store' folder into backend/data/vector_store before running the API.")


Vector store persisted at: ../data/vector_store
Config saved: {'chunk_size': 800, 'chunk_overlap': 100, 'embedding_model': 'all-MiniLM-L6-v2', 'ollama_model': 'llama3.2', 'collection_name': 'ml_docs'}

Copy the 'data/vector_store' folder into backend/data/vector_store before running the API.
